# Your Title Here

**Name(s)**: Austin Li, Yanhao Wu

**Website Link**: (your website link)

In [58]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'

# from dsc80_utils import * # Feel free to uncomment and use this.

## Step 1: Introduction

In [59]:
recipes = Path('RAW_recipes.csv')
interactions = Path('interactions.csv')
recipes_df = pd.read_csv(recipes)
interactions_df = pd.read_csv(interactions)
#Left merge the recipes and interactions datasets together.
df = recipes_df.merge(interactions_df, how = 'left', left_on = 'id', right_on = 'recipe_id')
#In the merged dataset, fill all ratings of 0 with np.nan
df['rating'] = df['rating'].replace(0, np.nan)
#Find the average rating per recipe, as a Series.
ratings_avgs = df.groupby('id')['rating'].mean().rename("Rating Avgs")
#Add this Series containing the average rating per recipe back to the recipes dataset 
df = df.merge(ratings_avgs, how = 'left', on = 'id')
df.head()

,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients,user_id,recipe_id,date,rating,review,Rating Avgs
0,1 brownies in the world best ever,333281,40,985201,2008-10-27,"['60-minutes-or-less', 'time-to-make', 'course...","[138.4, 10.0, 50.0, 3.0, 3.0, 19.0, 6.0]",10,['heat the oven to 350f and arrange the rack i...,"these are the most; chocolatey, moist, rich, d...","['bittersweet chocolate', 'unsalted butter', '...",9,386585.0,333281.0,2008-11-19,4.0,"These were pretty good, but took forever to ba...",4.0
1,1 in canada chocolate chip cookies,453467,45,1848091,2011-04-11,"['60-minutes-or-less', 'time-to-make', 'cuisin...","[595.1, 46.0, 211.0, 22.0, 13.0, 51.0, 26.0]",12,"['pre-heat oven the 350 degrees f', 'in a mixi...",this is the recipe that we use at my school ca...,"['white sugar', 'brown sugar', 'salt', 'margar...",11,424680.0,453467.0,2012-01-26,5.0,Originally I was gonna cut the recipe in half ...,5.0
2,412 broccoli casserole,306168,40,50969,2008-05-30,"['60-minutes-or-less', 'time-to-make', 'course...","[194.8, 20.0, 6.0, 32.0, 22.0, 36.0, 3.0]",6,"['preheat oven to 350 degrees', 'spray a 2 qua...",since there are already 411 recipes for brocco...,"['frozen broccoli cuts', 'cream of chicken sou...",9,29782.0,306168.0,2008-12-31,5.0,This was one of the best broccoli casseroles t...,5.0
3,412 broccoli casserole,306168,40,50969,2008-05-30,"['60-minutes-or-less', 'time-to-make', 'course...","[194.8, 20.0, 6.0, 32.0, 22.0, 36.0, 3.0]",6,"['preheat oven to 350 degrees', 'spray a 2 qua...",since there are already 411 recipes for brocco...,"['frozen broccoli cuts', 'cream of chicken sou...",9,1196280.0,306168.0,2009-04-13,5.0,I made this for my son's first birthday party ...,5.0
4,412 broccoli casserole,306168,40,50969,2008-05-30,"['60-minutes-or-less', 'time-to-make', 'course...","[194.8, 20.0, 6.0, 32.0, 22.0, 36.0, 3.0]",6,"['preheat oven to 350 degrees', 'spray a 2 qua...",since there are already 411 recipes for brocco...,"['frozen broccoli cuts', 'cream of chicken sou...",9,768828.0,306168.0,2013-08-02,5.0,Loved this. Be sure to completely thaw the br...,5.0


## Step 2: Data Cleaning and Exploratory Data Analysis

In [60]:
#Trying to answer the question: Do high-protein recipes get rated differently than low-protein recipes?
#Cleaning process: create new columns, specifically calorie to protein count
df['nutrition_list'] = df['nutrition'].str.strip('[]').str.split(',')
df['Calories'] = df['nutrition_list'].apply(lambda nut: float(nut[0]))
df['CP ratio'] = df['nutrition_list'].apply(lambda nut: np.nan if float(nut[-3]) == 0 else float(nut[0]) / float(nut[-3]))
df = df.dropna(subset = 'CP ratio')

#CP ratio to categorical, a CP ration below 15 will be considered high for this
df['CP classification'] = df['CP ratio'].apply(lambda val: 'High protein' if val <= 15 else 'Low Protein')
df.head()

,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,...,user_id,recipe_id,date,rating,review,Rating Avgs,nutrition_list,Calories,CP ratio,CP classification
0,1 brownies in the world best ever,333281,40,985201,2008-10-27,"['60-minutes-or-less', 'time-to-make', 'course...","[138.4, 10.0, 50.0, 3.0, 3.0, 19.0, 6.0]",10,['heat the oven to 350f and arrange the rack i...,"these are the most; chocolatey, moist, rich, d...",...,386585.0,333281.0,2008-11-19,4.0,"These were pretty good, but took forever to ba...",4.0,"[138.4, 10.0, 50.0, 3.0, 3.0, 19.0, 6.0]",138.4,46.133333,Low Protein
1,1 in canada chocolate chip cookies,453467,45,1848091,2011-04-11,"['60-minutes-or-less', 'time-to-make', 'cuisin...","[595.1, 46.0, 211.0, 22.0, 13.0, 51.0, 26.0]",12,"['pre-heat oven the 350 degrees f', 'in a mixi...",this is the recipe that we use at my school ca...,...,424680.0,453467.0,2012-01-26,5.0,Originally I was gonna cut the recipe in half ...,5.0,"[595.1, 46.0, 211.0, 22.0, 13.0, 51.0, 2...",595.1,45.776923,Low Protein
2,412 broccoli casserole,306168,40,50969,2008-05-30,"['60-minutes-or-less', 'time-to-make', 'course...","[194.8, 20.0, 6.0, 32.0, 22.0, 36.0, 3.0]",6,"['preheat oven to 350 degrees', 'spray a 2 qua...",since there are already 411 recipes for brocco...,...,29782.0,306168.0,2008-12-31,5.0,This was one of the best broccoli casseroles t...,5.0,"[194.8, 20.0, 6.0, 32.0, 22.0, 36.0, 3.0]",194.8,8.854545,High protein
3,412 broccoli casserole,306168,40,50969,2008-05-30,"['60-minutes-or-less', 'time-to-make', 'course...","[194.8, 20.0, 6.0, 32.0, 22.0, 36.0, 3.0]",6,"['preheat oven to 350 degrees', 'spray a 2 qua...",since there are already 411 recipes for brocco...,...,1196280.0,306168.0,2009-04-13,5.0,I made this for my son's first birthday party ...,5.0,"[194.8, 20.0, 6.0, 32.0, 22.0, 36.0, 3.0]",194.8,8.854545,High protein
4,412 broccoli casserole,306168,40,50969,2008-05-30,"['60-minutes-or-less', 'time-to-make', 'course...","[194.8, 20.0, 6.0, 32.0, 22.0, 36.0, 3.0]",6,"['preheat oven to 350 degrees', 'spray a 2 qua...",since there are already 411 recipes for brocco...,...,768828.0,306168.0,2013-08-02,5.0,Loved this. Be sure to completely thaw the br...,5.0,"[194.8, 20.0, 6.0, 32.0, 22.0, 36.0, 3.0]",194.8,8.854545,High protein


In [61]:
#look at some columns, ratings distribution for example
fig = px.histogram(df, x="rating", nbins = 5)
fig.update_traces(marker_line_color='white', marker_line_width=1)
fig

In [62]:
fig = px.histogram(df, x='rating', color='CP classification', barmode='group')
fig


## Step 3: Assessment of Missingness

In [63]:
#rating is likely a NMAR column, as the missingness in the column likely depends on the value itself. A missing rating comes from a rating of 0, which could signify that the user didn't like the recipe enough to rate it.

#testing rating on different columns
# def perm_test_rating(df, col):
#     df = df.copy()
#     df['rating_is_na'] = df['rating'].isna().astype(int)
#     diffs = []
#     obs_diff = np.abs(df.groupby('rating_is_na')[col].mean().iloc[0] - df.groupby('rating_is_na')[col].mean().iloc[1])
#     for _ in range(10000):
#         df['rating_is_na'] = np.random.permutation(df['rating_is_na'])
#         test_diff = np.abs(df.groupby('rating_is_na')[col].mean().iloc[0] - df.groupby('rating_is_na')[col].mean().iloc[1])
#         diffs.append(test_diff)
#     return (diffs >= obs_diff).mean()
# perm_test_rating(df, 'n_steps')

#permutation test on minutes gives a p-value of 0.11, not statistically significant, so the missingness of the rating column likely doesn't depend on minutes
#permutation test on n_steps gives a p-value of 0.0, which is statistcally significant, so the missingness of the ratings columns could depend on the steps in the recipe

## Step 4: Hypothesis Testing

In [64]:
# H_0: High-protein recipes do not get rated differently than low-protein recipes
# H_1: High-protein recipes do get rated differently than low-protein recipes

#Test statistic: diff of means
obs_test_stat = np.abs(df.groupby('CP classification')['rating'].mean().iloc[0] - df.groupby('CP classification')['rating'].mean().iloc[1])

# Testing at a significance level of 0.05
# df_copy = df.copy()
# diffs = []
# for _ in range(10000):
#     df_copy['CP classification'] = np.random.permutation(df['CP classification'])
#     diffs.append(np.abs(df_copy.groupby('CP classification')['rating'].mean().iloc[0] - df_copy.groupby('CP classification')['rating'].mean().iloc[1]))
# (diffs >= obs_test_stat).mean()

#we see a value of 0.0, which suggests that there is a significant difference between the ratings of high protein recipes and low protein recipes

## Step 5: Framing a Prediction Problem

In [ ]:
# Predicting ratings
df.columns

Index(['name', 'id', 'minutes', 'contributor_id', 'submitted', 'tags',
       'nutrition', 'n_steps', 'steps', 'description', 'ingredients',
       'n_ingredients', 'user_id', 'recipe_id', 'date', 'rating', 'review',
       'Rating Avgs', 'nutrition_list', 'Calories', 'CP ratio',
       'CP classification'],
      dtype='object')

## Step 6: Baseline Model

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
# Features: steps, ingredients, calories

lr_df = df.drop_duplicates(subset = 'id')
X = lr_df[['n_steps', 'n_ingredients', 'Calories']]
y = np.log1p(lr_df['minutes'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2) # We don't have to choose 0.25.

pl = Pipeline([('lin-reg', LinearRegression())])
pl.fit(X_train, y_train)
print(pl.score(X_test, y_test))
root_mean_squared_error(y_test, pl.predict(X_test))


0.1579519215471259


np.float64(0.9270505975626764)

## Step 7: Final Model

In [90]:
# Try binarizing steps as well
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import Binarizer

from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

errs_df = pd.DataFrame()
kf = KFold(5, shuffle=True, random_state=42)
for threshold in range(0, 20, 1):
    preproc = ColumnTransformer([
        ('bin_steps', Binarizer(threshold=threshold), ['n_steps']),
        ('keep_steps', 'passthrough', ['n_steps']),
    ], remainder='passthrough') 

    new_pl = Pipeline([
        ('preproc', preproc),
        ('lin-reg', LinearRegression())
    ])  
    
    errs = cross_val_score(new_pl, X_train, y_train, 
                cv=kf, scoring='neg_root_mean_squared_error')
    errs_df[f'threshold {threshold}'] = -errs
    # new_pl.fit(X_train, y_train)    
    # R_sq = new_pl.score(X_test, y_test)
    # RMSE = root_mean_squared_error(y_test, new_pl.predict(X_test))
    # temp.append((threshold, R_sq, RMSE))
# threshold_hyperparam_df = pd.DataFrame(temp, columns = ['Hyperparam', 'R_sq', 'RMSE'])
# threshold_hyperparam_df
errs_df.index = [f'Fold {i}' for i in range(1, 6)]
errs_df.index.name = 'Validation Fold'
errs_df

,threshold 0,threshold 1,threshold 2,threshold 3,threshold 4,threshold 5,threshold 6,threshold 7,threshold 8,threshold 9,threshold 10,threshold 11,threshold 12,threshold 13,threshold 14,threshold 15,threshold 16,threshold 17,threshold 18,threshold 19
Validation Fold,,,,,,,,,,,,,,,,,,,,
Fold 1,0.942146,0.938973,0.932313,0.929217,0.929346,0.933171,0.936815,0.938698,0.940207,0.941614,0.942056,0.942090,0.941763,0.941233,0.940662,0.940357,0.940056,0.939586,0.939845,0.940076
Fold 2,0.925605,0.922351,0.915377,0.911753,0.913183,0.917379,0.920792,0.922969,0.924304,0.925343,0.925768,0.925503,0.925102,0.924704,0.924683,0.924445,0.924115,0.923508,0.923684,0.923463
Fold 3,0.912406,0.907984,0.902140,0.899762,0.901994,0.904737,0.907840,0.910017,0.911349,0.912090,0.912388,0.912325,0.911981,0.911833,0.911283,0.910991,0.911435,0.911637,0.911018,0.910987
Fold 4,0.906975,0.902566,0.898075,0.895858,0.896559,0.897654,0.900438,0.903197,0.905235,0.906110,0.906872,0.907213,0.907042,0.906570,0.905925,0.905402,0.905246,0.905093,0.904615,0.904545
Fold 5,0.939136,0.932978,0.926700,0.925461,0.928915,0.930910,0.934311,0.937133,0.938107,0.938602,0.939149,0.939020,0.938670,0.938452,0.938544,0.938390,0.938465,0.937734,0.937329,0.937031


In [91]:
errs_df.mean().idxmin()

'threshold 3'

In [ ]:
#Adding in tags, hyperparameter is what tags/how many to look for
# import ast
# df['tags'].apply(ast.literal_eval).explode().unique()

#scrolling through list: ['course', 'main-ingredient', 'preparation', 'lunch', 'snacks', 'number-of-servings', 'side-dishes', 'easy', 'inexpensive']
import ast
lr_df = df.drop_duplicates(subset = 'id')
for tag in ['course', 'main-ingredient', 'preparation', 'lunch', 'snacks', 'number-of-servings', 'side-dishes', 'easy', 'inexpensive']:
    lr_df[f'one_hot_tag_{tag}'] = lr_df['tags'].apply(lambda tags: tag in ast.literal_eval(tags)).astype(int)
    X_train_new = lr_df.loc[X_train.index, new_cols]
    X_test_new  = lr_df.loc[X_test.index, new_cols]

array(['60-minutes-or-less', 'time-to-make', 'course', 'main-ingredient',
       'preparation', 'for-large-groups', 'desserts', 'lunch', 'snacks',
       'cookies-and-brownies', 'chocolate', 'bar-cookies', 'brownies',
       'number-of-servings', 'cuisine', 'north-american', 'canadian',
       'british-columbian', 'side-dishes', 'vegetables', 'easy',
       'beginner-cook', 'broccoli', 'occasion', 'american',
       'southern-united-states', 'dinner-party', 'holiday-event', 'cakes',
       'dietary', 'christmas', 'thanksgiving', 'low-sodium',
       'low-in-something', 'taste-mood', 'sweet', '4-hours-or-less',
       'main-dish', 'potatoes', 'meatloaf', 'simply-potatoes2',
       'weeknight', '30-minutes-or-less', 'beef', 'diabetic',
       'kid-friendly', 'stove-top', 'comfort-food', 'inexpensive',
       'ground-beef', 'meat', 'greens', 'lettuces', 'tomatoes',
       'equipment', '3-steps-or-less', 'soups-stews', 'beans', 'pork',
       'mexican', 'stews', 'crock-pot-slow-cooker', 's

In [93]:
#threshold 3 is the best
preproc = ColumnTransformer([
        ('bin_steps', Binarizer(threshold=3), ['n_steps']),
        ('keep_steps', 'passthrough', ['n_steps']),
    ], remainder='passthrough') 
new_pl = Pipeline([
        ('preproc', preproc),
        ('lin-reg', LinearRegression())
    ])  

new_pl.fit(X_train, y_train)
print(new_pl.score(X_test, y_test))
root_mean_squared_error(y_test, new_pl.predict(X_test))

0.179900265280049


np.float64(0.9148888385367864)

## Step 8: Fairness Analysis

In [ ]:
# TODO